In [5]:
import pandas as pd
from sklearn.metrics import confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar
import pandas as pd

In [6]:
indo = pd.read_csv("hasil_indobert.csv")
roberta = pd.read_csv("hasil_roberta.csv")
inset = pd.read_csv("hasil_inset.csv")

In [7]:
indo.head()

,y_test,y_pred_indobert
0,negative,negative
1,negative,negative
2,positive,negative
3,positive,positive
4,negative,neutral


In [8]:
y_test = indo["y_test"]

y_pred_indobert = indo["y_pred_indobert"]
y_pred_roberta = roberta["y_pred_roberta"]
y_pred_inset = inset["y_pred_inset"]

In [9]:
print(len(y_test))
print(len(y_pred_indobert))
print(len(y_pred_roberta))
print(len(y_pred_inset))

1757
1757
1757
1757


In [10]:
import numpy as np

def mcnemar_test(y_true, pred1, pred2, model1_name, model2_name):

    # Correct / Incorrect prediction
    model1_correct = (pred1 == y_true)
    model2_correct = (pred2 == y_true)

    # Contingency table
    both_correct = np.sum(model1_correct & model2_correct)
    model1_only = np.sum(model1_correct & ~model2_correct)
    model2_only = np.sum(~model1_correct & model2_correct)
    both_wrong = np.sum(~model1_correct & ~model2_correct)

    table = [
        [both_correct, model1_only],
        [model2_only, both_wrong]
    ]

    result = mcnemar(table, exact=False, correction=True)

    print("="*50)
    print(f"{model1_name} vs {model2_name}")
    print("="*50)
    print("Contingency Table")
    print(pd.DataFrame(
        table,
        index=[f"{model1_name} Correct", f"{model1_name} Wrong"],
        columns=[f"{model2_name} Correct", f"{model2_name} Wrong"]
    ))

    print(f"\nStatistic : {result.statistic:.4f}")
    print(f"P-value   : {result.pvalue:.6f}")

    if result.pvalue < 0.05:
        print("Result    : Significant Difference")
    else:
        print("Result    : No Significant Difference")

    return result

In [11]:
mcnemar_test(
    y_test,
    y_pred_indobert,
    y_pred_roberta,
    "IndoBERT",
    "RoBERTa"
)

IndoBERT vs RoBERTa
Contingency Table
                  RoBERTa Correct  RoBERTa Wrong
IndoBERT Correct             1043            114
IndoBERT Wrong                169            431

Statistic : 10.3039
P-value   : 0.001328
Result    : Significant Difference


<bunch containing results, print to see contents>

In [12]:
mcnemar_test(
    y_test,
    y_pred_indobert,
    y_pred_inset,
    "IndoBERT",
    "InSet"
)

IndoBERT vs InSet
Contingency Table
                  InSet Correct  InSet Wrong
IndoBERT Correct            857          300
IndoBERT Wrong              243          357

Statistic : 5.7753
P-value   : 0.016253
Result    : Significant Difference


<bunch containing results, print to see contents>

In [13]:
mcnemar_test(
    y_test,
    y_pred_roberta,
    y_pred_inset,
    "RoBERTa",
    "InSet"
)

RoBERTa vs InSet
Contingency Table
                 InSet Correct  InSet Wrong
RoBERTa Correct            898          314
RoBERTa Wrong              202          343

Statistic : 23.8779
P-value   : 0.000001
Result    : Significant Difference


<bunch containing results, print to see contents>